# <div align="center"><b> Recortes </b></div>

<div align="right">

<!-- [![Binder](http://mybinder.org/badge.svg)](https://mybinder.org/) -->
[![nbviewer](https://img.shields.io/badge/render-nbviewer-orange?logo=Jupyter)](https://nbviewer.org)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://githubtocolab.com)

</div>

* * *

<style>
/* Limitar la altura de las celdas de salida en html */
.jp-OutputArea.jp-Cell-outputArea {
    max-height: 500px;
}
</style>

🛻 <em><font color='MediumSeaGreen'>  Instalaciones: </font></em> 🛻

Este notebook utiliza [Poetry](https://python-poetry.org/) para la gestión de dependencias.
Primero instala Poetry siguiendo las instrucciones de su [documentación oficial](https://python-poetry.org/docs/#installation).
Luego ejecuta el siguiente comando para instalar las dependencias necesarias y activar el entorno virtual:

- Bash:
```bash
poetry install
eval $(poetry env activate)
```

- PowerShell:
```powershell
poetry install
Invoke-Expression (poetry env activate)
```

<!-- Descargar archivos adicionales:
!gdown https://drive.google.com/drive/folders/1UBZ8PEbtmiWMGkULu7GAt3VhUpeTy9l7?usp=sharing --folder -->

✋ <em><font color='DodgerBlue'>Importaciones:</font></em> ✋

In [ ]:
# Recarga automática de módulos en Jupyter Notebook
%reload_ext autoreload
%autoreload 2

import sys, json, requests, os, shutil, yaml
from pathlib import Path
from pprint import pprint

from loguru import logger as LOGGER
from modulo_ia.config import config as CONFIG
from modulo_utilidades.database_comunication.mongodb_client import mongodb as MONGODB

# os.environ["ALBUMENTATIONS_DISABLE"] = "1" # Deshabilita Albumentations por defecto.
import ultralytics
from ultralytics import YOLO, settings
from ultralytics.data.utils import visualize_image_annotations
import torch

import mlflow
import cv2
import PIL
from PIL import Image
import pandas as pd

import modulo_ia.dataset as DatasetProcessor
import modulo_ia.features as FeaturesProcessor
import modulo_ia.utils.gpu as GpuUtils
import modulo_ia.utils.yolo_utils as YoloUtils
from modulo_ia.modeling.predict import DetectionModelPredictor
from modulo_ia.utils.types import DatasetFormat

import modulo_utilidades.labeling.procesador_anotaciones_coco_dataset as CocoDatasetUtils
import modulo_utilidades.s3_comunication.procesador_s3 as ProcesadorS3
import modulo_utilidades.labeling.visualizador_coco_dataset as VisualizadorCocoDataset
import modulo_utilidades.labeling.procesador_anotaciones_mongodb as ProcesadorAnotacionesMongoDB
import modulo_utilidades.labeling.procesador_geojson_kml as ProcesadorGeoJSONKML

import fiftyone as fo

🔧 <em><font color='tomato'>Configuraciones:</font></em> 🔧


In [11]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"  # Establece el dispositivo.
LOGGER.remove()
LOGGER.add(sys.stderr, level="INFO")
PIL.Image.MAX_IMAGE_PIXELS = None

# Parámetros
SHOULD_PROCESS_DATASET = False  # Descarga el dataset completo
SHOULD_TRAIN = False  # Entrenamiento
BATCH_SIZE = 64  # 16 -> Yolo11x | # 64 -> Yolo11n
N_EPOCHS = 200  # Número de épocas
VERBOSE = True  # Muestra época a época la evolución
IMG_SIZE = 640  # Tamaño de la imagen
RANDOM_SEED = CONFIG.seed  # Semilla para la aleatoriedad

RAW_DATA_FOLDER = CONFIG.folders.raw_data_folder
EXTERNAL_DATA_FOLDER = CONFIG.folders.external_data_folder
INTERIM_DATA_FOLDER = CONFIG.folders.interim_data_folder
PROCESSED_DATA_FOLDER = CONFIG.folders.processed_data_folder
DATA_FOLDER = CONFIG.folders.data_folder

DOWNLOAD_PREDICTION_FOLDER = Path("downloads") / "predictions"  # Carpeta para descargar parches de prueba
# AUMENTATIONS_CONFIG_FILE_PATH = Path("train_default_aumentations.yaml")
AUMENTATIONS_CONFIG_FILE_PATH = Path("train_custom1_aumentations.yaml")

TASK_NAME = "palm_detection"  # Nombre de la tarea
MODEL_NAME = "yolo11n"

DATASET_NAME = CONFIG.names.palm_dataset_name
DATASET_VERSION = CONFIG.versions.palm_dataset_name
DATASET_IDENTIFIER = f"{DATASET_NAME}_{DATASET_VERSION}"

DATASET_FINAL_FORMAT = CONFIG.datasets_processed_format.yolo  # Formato final del dataset procesado

DATASET_RAW_NAME = DATASET_IDENTIFIER
DATASET_RAW_PATH = RAW_DATA_FOLDER / DATASET_RAW_NAME
DATASET_INTERIM_NAME = DATASET_IDENTIFIER
DATASET_INTERIM_PATH = INTERIM_DATA_FOLDER / DATASET_INTERIM_NAME
DATASET_INTERIM_STEP_PATH = INTERIM_DATA_FOLDER / f"{DATASET_INTERIM_NAME}_step"
DATASET_INTERIM_STEP_NAME = f"{DATASET_INTERIM_NAME}_step"
DATASET_PROCESSED_NAME = f"{DATASET_IDENTIFIER}_{DATASET_FINAL_FORMAT}"
DATASET_PROCESSED_PATH = PROCESSED_DATA_FOLDER / DATASET_PROCESSED_NAME

EXPERIMENT_NAME = (
    f"{DATASET_IDENTIFIER}_{TASK_NAME}_{MODEL_NAME.replace('/', '_')}_{IMG_SIZE}"  # Nombre del experimento
)
MODEL_FOLDER = CONFIG.folders.models_folder / EXPERIMENT_NAME
MODEL_CHECKPOINT = f"{MODEL_NAME}.pt"

CUSTOM_TRANSFORMS = [
    "crop",
    "balance",
    str(AUMENTATIONS_CONFIG_FILE_PATH).replace(".yaml", ""),
]  # Transformaciones personalizadas

CLASS_NAMES = {
    0: "palmera"
}

CATEGORIES = [{"id": id, "name": name, } for id, name in CLASS_NAMES.items()]  # Categorías del dataset

# Configuraciones de MLflow
MLFLOW_URL = CONFIG.mlflow.tracking_uri
os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_URL  # Configura la URI de seguimiento de MLflow.
os.environ["MLFLOW_EXPERIMENT_NAME"] = EXPERIMENT_NAME  # Configura el nombre del experimento de MLflow.
os.environ["MLFLOW_TAGS"] = (
    '{"model_family": "palm_detection", "model_version": "v1.0"}'  # Configura las etiquetas del experimento de MLflow.
)
settings.update({"mlflow": True})  # Habilita el uso de MLflow en ultralytics

ultralytics.checks()  # Verifica la instalación de ultralytics
LOGGER.info(f"Dispositivo actual: {DEVICE}")

Ultralytics 8.3.161  Python-3.13.3 torch-2.7.1+cu128 CUDA:0 (NVIDIA GeForce RTX 4080 SUPER, 16376MiB)
Setup complete  (20 CPUs, 127.9 GB RAM, 1415.9/1862.9 GB disk)


2025-08-12 12:46:48.128 | INFO     | __main__:<module>:71 - Dispositivo actual: cuda


<div align="center">✨Datos del proyecto:✨</div>

<p></p>

<div align="center">

| **Subtitulo**   | Exploración de los datos                                                                                                        |
| --------------- | -------------------------------------------------------------------------------------------------------------------------------------- |
| **Descrpción**  | <small>Notebook de exploración de los datos</small>                                                                    |

</div>

# Chequeo de conexiones

In [12]:
# Chequeo de conexión a MinIO
LOGGER.info("Chequeando conexión a MinIO...")
ProcesadorS3.test_connection()

# Chequeo de conexión a MongoDB
LOGGER.info("Chequeando conexión a MongoDB...")
MONGODB.command("ping")  # Verifica la conexión a MongoDB
LOGGER.success("Conexión a MongoDB verificada correctamente.")

2025-08-12 12:46:48.258 | INFO     | __main__:<module>:2 - Chequeando conexión a MinIO...
2025-08-12 12:46:48.273 | SUCCESS  | modulo_apps.s3_comunication.procesador_s3:test_connection:37 - Conexión exitosa a S3 y acceso al bucket 'picudo-rojo-bucket' verificado.
2025-08-12 12:46:48.274 | INFO     | __main__:<module>:6 - Chequeando conexión a MongoDB...
2025-08-12 12:46:48.276 | SUCCESS  | __main__:<module>:8 - Conexión a MongoDB verificada correctamente.


## Descarga del dataset

In [13]:
if SHOULD_PROCESS_DATASET:
    LOGGER.info("Descargando el dataset completo...")
    DatasetProcessor.download_full_raw_dataset(output_folder=DATASET_RAW_PATH)

if DATASET_RAW_PATH.exists():
    LOGGER.info("Dataset ya descargado. Procesando dataset...")
    dataset_metrics = DatasetProcessor.get_dataset_metrics(
        dataset_path=DATASET_RAW_PATH, dataset_name=DATASET_RAW_NAME, dataset_format=DatasetProcessor.DatasetFormat.COCO
    )
    dataset_stats = DatasetProcessor.get_dataset_stats(
        dataset_path=DATASET_RAW_PATH, dataset_name=DATASET_RAW_NAME, dataset_format=DatasetProcessor.DatasetFormat.COCO
    )
    LOGGER.info(f"Metricas del dataset:\n {json.dumps(dataset_metrics, indent=2)}")
    LOGGER.info(f"Estadísticas del dataset:\n {json.dumps(dataset_stats, indent=2)}")

2025-08-12 12:46:48.406 | INFO     | __main__:<module>:6 - Dataset ya descargado. Procesando dataset...


 100% |███████████████████| 99/99 [3.0s elapsed, 0s remaining, 30.4 samples/s]      
 100% |███████████████████| 99/99 [3.1s elapsed, 0s remaining, 31.8 samples/s]      


2025-08-12 12:46:54.583 | INFO     | __main__:<module>:13 - Metricas del dataset:
 {
  "total_count": 99,
  "class_count": {
    "palmera-exterminada": 40,
    "palmera-infectada": 57,
    "palmera-muerta": 110,
    "palmera-sana": 5670
  }
}
2025-08-12 12:46:54.583 | INFO     | __main__:<module>:14 - Estadísticas del dataset:
 {
  "samples_count": 99,
  "samples_bytes": 1354982,
  "samples_size": "1.3MB",
  "media_bytes": 0,
  "media_size": "0.0B",
  "total_bytes": 1354982,
  "total_size": "1.3MB"
}


## Visualizamos el dataset

In [ ]:
DatasetProcessor.copy_dataset_to_quality(RAW_DATA_FOLDER, DATASET_RAW_NAME)

 100% |███████████████████| 99/99 [3.0s elapsed, 0s remaining, 32.0 samples/s]      
